# 10. Ders: His Kavanozunu Geliştirmek (Türkçe Model Entegrasyonu)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Geçen dersimizde kurduğumuz His Kavanozu (MoodJar) uygulaması İngilizce'de harikalar yaratıyordu. Ancak ona "Bugün çok üzgünüm" dediğimizde bize mantıksız bir cevap verdi. Bunun sebebi, yapay zekanın aslında İngilizceyi veya Türkçeyi bilmemesi, sadece kendisine eğitim sırasında verilen sayılar arasındaki istatistiksel ilişkileri (kelime vektörlerini) bilmesidir. Hiç Fransızca duymamış birine Fransızca bir şiir okursanız sadece sesler duyar, anlam çıkaramaz.

Bu dersimizde, daha önce Türk mühendisler veya uluslararası ekipler tarafından milyonlarca Türkçe cümle ile eğitilmiş açık kaynaklı (ücretsiz) bir yapay zeka modelini Hugging Face'ten bulup sistemimize entegre edeceğiz.

## Adım 2: Teknik Terimler ve Sözlük

- Model Hub (Model Merkezi): Hugging Face gibi platformlarda on binlerce farklı yapay zeka modelinin bulunduğu açık kaynaklı kütüphane/havuz.
- Multilingual Model (Çok Dilli Model): Sadece tek bir dilde değil, aynı anda 100 farklı dilde eğitim almış ve diller arası çeviri mantığını kavramış kapsayıcı modeller.
- BERTurk: Sadece Türkçe metinler (Vikipedi, haberler, forumlar) kullanılarak eğitilmiş meşhur bir dil modelinin adıdır.
- Fine-Tuning (İnce Ayar): Genel bir modeli (örneğin sadece Türkçe bilen bir model) alıp, ona sadece 'Duygu Analizi' yapması için ekstra küçük bir eğitim verme işlemidir.

## Adım 3: Hugging Face'te Türkçe Model Arama Stratejisi

Hugging Face'in resmi sitesine (huggingface.co) girip arama çubuğuna 'Turkish Sentiment Analysis' yazdığınızda karşınıza yüzlerce model çıkar. En çok indirilen ve en yüksek beğeni alan modeller güvenilirdir. Bu dersimizde `savasy/bert-base-turkish-sentiment-cased` isimli popüler Türkçe modeli kullanacağız.

In [ ]:
# İlk olarak transformers kütüphanemizi ekliyoruz.
from transformers import pipeline

print("Kütüphane hazır. Şimdi özel Türkçe modelimizi indireceğiz...")

## Adım 4: Pipeline İçine Özel Model Parametresi Eklemek

Daha önce sadece `pipeline("sentiment-analysis")` diyerek varsayılan İngilizce modeli çağırmıştık. Şimdi parantezin içine `model="..."` yazarak spesifik bir Türkçe model indireceğiz.

*Not: Bu işlem Colab sunucusunun hızına bağlı olarak 10-20 saniye sürebilir.*

In [ ]:
# Hugging Face'te bulunan Savaş Yıldırım'a ait açık kaynaklı Türkçe modeli indiriyoruz.
turkce_duygu_modeli = pipeline("sentiment-analysis", model="savasy/bert-base-turkish-sentiment-cased")

print("Türkçe yapay zeka modeli (BERTurk tabanlı) başarıyla belleğe yüklendi!")

## Adım 5: Türkçe Modeli Test Etme

Şimdi önceki derste hata aldığımız o meşhur cümleyi tekrar deneyelim ve Türkçeyi ne kadar iyi anladığını görelim.

In [ ]:
deneme_cumlesi = "Bugün her şey çok ters gitti, gerçekten çok üzgünüm."

sonuc = turkce_duygu_modeli(deneme_cumlesi)
print("Modelin Analizi:")
print(sonuc)

# Çıktıda 'negative' ve ona ait yüksek bir skor görmeliyiz.

## Adım 6: Özel Etiketleri Anlamak (Label Mapping)

Her modeli farklı bir mühendis eğittiği için, çıktıların etiketleri (label) değişebilir. Biri 'POSITIVE' derken, diğeri 'olumlu', bir başkası ise 'LABEL_1' diyebilir. Bizim kullandığımız model çıktı olarak "positive" ve "negative" dönmektedir. Bunu sistemimize bağlamak için küçük bir sözlük ayıklaması (ayıklama işlemi) yapacağız.

In [ ]:
# Ayrıştırma işlemi
etiket = sonuc[0]['label']
skor = sonuc[0]['score']

print(f"Anlaşılan Duygu: {etiket.upper()}")
print(f"Eminlik Oranı: {skor:.2f} (Yüzde {skor*100:.0f})")

## Adım 7: İngilizce ve Türkçe Modelin Karşılaştırılması

İki modeli aynı anda yükleyip, ikisine de aynı Türkçe kelimeyi verdiğimizde aradaki farkı görmek, makine öğrenmesi modellerinin eğitim setine ne kadar bağımlı (biased) olduğunu kanıtlar.

In [ ]:
# İngilizce modeli tekrar yüklüyoruz (Zaten önceden inmişti, hızlıca yüklenir)
ingilizce_model = pipeline("sentiment-analysis")

test_kelimesi = "Mükemmel"

ingilizce_sonuc = ingilizce_model(test_kelimesi)
turkce_sonuc = turkce_duygu_modeli(test_kelimesi)

print("Test Edilen Kelime:", test_kelimesi)
print("İngilizce Modelin Kararı:", ingilizce_sonuc[0]['label'])
print("Türkçe Modelin Kararı:", turkce_sonuc[0]['label'])

# İngilizce model 'Mükemmel' kelimesini bilmediği için ona büyük ihtimalle yanlış bir etiket atayacaktır.

## Adım 8: Dene ve Keşfet - Yapay Zeka İroniyi Anlar mı?

Sıradan `if-else` kodları ironiyi (kinayeyi) asla anlayamaz. Çünkü cümlenin içinde 'harika' gibi pozitif kelimeler geçer ama genel anlam negatiftir. Bakalım indirdiğimiz derin öğrenme modeli bunu anlayabilecek mi?

In [ ]:
ironi_cumlesi = "Telefonu o kadar harika yapmışlar ki, daha şarja takmadan bataryası bitiyor."

ironi_sonuc = turkce_duygu_modeli(ironi_cumlesi)
print("İroni Cümlesi:", ironi_cumlesi)
print("Modelin Kararı:", ironi_sonuc[0]['label'], "- Eminlik:", round(ironi_sonuc[0]['score'], 2))

# Eğer model bunu 'negative' olarak sınıflandırdıysa, sadece kelimeleri saymadığını, 
# cümlenin bağlamını (context) anladığını ispatlamış oluruz.

## Adım 9: Sıra Sende Görevi - Türkçe Haber Başlığı Analizörü

Problem: Bir haber sitesinde çalışıyorsunuz. Editörlerin girdiği haber başlıklarının olumlu mu yoksa olumsuz mu olduğunu otomatik etiketleyen bir sistem kurmalısınız.

Gereksinimler:
1. `haber_basliklari` adında bir liste oluşturun. İçine en az 4 tane Türkçe haber başlığı yazın (2 olumlu, 2 olumsuz).
2. Bir `for` döngüsü ile listenin içinde gezinin.
3. Her bir başlığı `turkce_duygu_modeli`'ne verin.
4. Sonucu ayrıştırın (etiket ve skor).
5. Çıktı olarak şu formata benzeyen bir metin yazdırın:
   `Başlık: "Borsa haftaya rekorla başladı" -> Durum: POSITIVE (Güven: %95)`

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 10. Dersi tamamlayarak global yapay zeka modellerini yerelleştirmeyi (Türkçe dil desteği eklemeyi) başardınız. Artık MoodJar projemizin kalbi tamamen Türkçe atıyor.

Bir sonraki dersimizde, bu beyni alıp sürekli uyanık kalan, bizi dinleyen ve bizimle empati kuran bir 'Diyalog Asistanına' dönüştüreceğiz.